# Duality Gap Toy Problem

Chapter 12: Optimization Methods · *Mathematics for AI and Machine Learning* by Fuheng Wu

Visualize primal and dual objectives and the effect of strong duality conditions.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-12/duality-gap/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's defaults: minimize 0.5 (x - c)^2 subject to x >= b, multiplier lam
c, b, lam_probe = 0.8, 1.5, 0.0

# Lagrangian L(x, lam) = 0.5 (x - c)^2 + lam (b - x), lam >= 0
# Minimizing over x gives x = c + lam, so the dual function is
#   g(lam) = -0.5 lam^2 + lam (b - c)


def dual(lam):
    return -0.5 * lam ** 2 + lam * (b - c)


x_star = max(c, b)                        # primal solution
p_star = 0.5 * (x_star - c) ** 2
lam_star = max(0.0, b - c)                # dual solution
d_star = dual(lam_star)
print(f"primal: x* = {x_star}, p* = {p_star:.6f}")
print(f"dual:   lam* = {lam_star:.6f}, d* = {d_star:.6f}")
print(f"duality gap p* - d* = {p_star - d_star:.2e} (strong duality: convex problem with a strictly feasible point)")

# Weak duality: every lam >= 0 gives a lower bound
for lam in (lam_probe, 0.3, lam_star, 1.2):
    print(f"g({lam:.2f}) = {dual(lam):+.6f} <= p* = {p_star:.6f}")

# KKT: stationarity x* - c - lam* = 0, complementary slackness lam* (b - x*) = 0
print("\nKKT stationarity x* - c - lam* =", x_star - c - lam_star, "| slackness lam*(b - x*) =", lam_star * (b - x_star))

# ── Check numerically: maximize g by gradient ascent ────────────────────────
lam = torch.tensor(0.0, dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([lam], lr=0.2)
for _ in range(200):
    opt.zero_grad()
    (-dual(lam)).backward()
    opt.step()
    with torch.no_grad():
        lam.clamp_(min=0.0)               # keep lam >= 0
print("gradient ascent on the dual: lam =", round(lam.item(), 6), " g(lam) =", round(dual(lam).item(), 6))

primal: x* = 1.5, p* = 0.245000
dual:   lam* = 0.700000, d* = 0.245000
duality gap p* - d* = 0.00e+00 (strong duality: convex problem with a strictly feasible point)
g(0.00) = +0.000000 <= p* = 0.245000
g(0.30) = +0.165000 <= p* = 0.245000
g(0.70) = +0.245000 <= p* = 0.245000
g(1.20) = +0.120000 <= p* = 0.245000

KKT stationarity x* - c - lam* = 0.0 | slackness lam*(b - x*) = 0.0


gradient ascent on the dual: lam = 0.7  g(lam) = 0.245
